# Exploração inicial do dataset de risco de obesidade

Este notebook reutiliza os componentes governados do projeto para:

1. validar ou importar o snapshot do Kaggle;
2. publicar ou reutilizar o snapshot imutável no MinIO;
3. ler o CSV diretamente do MinIO;
4. aplicar somente a normalização canônica do alvo em memória;
5. iniciar a análise exploratória sem alterar os dados raw.

> O dataset representa uma fotografia transversal e não um risco futuro. As análises são educacionais e não constituem diagnóstico médico.

## Pré-requisitos

A partir da raiz do projeto, instale o pacote e inicie o MinIO:

```bash
python -m pip install -r requirements.txt
python -m pip install -e .
docker compose up -d minio
```

Se o snapshot local ainda não existir, configure também a credencial Kaggle conforme `.env.example`. Os defaults do MinIO correspondem ao ambiente local definido no Compose.

In [ ]:
from __future__ import annotations

import sys
from io import BytesIO
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError('Não foi possível localizar a raiz do projeto.')


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
load_dotenv(PROJECT_ROOT / '.env')
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', context='notebook')
print(f'Projeto: {PROJECT_ROOT}')

## 1. Garantir o snapshot governado local

A ingestão é idempotente: se o CSV e o manifesto já existirem e estiverem íntegros, o Kaggle não é chamado. Se estiverem ausentes, o download ocorre em staging e só é publicado depois das validações.

In [ ]:
from obesity_risk_pipeline.config import load_ingestion_settings
from obesity_risk_pipeline.ingestion import KaggleApiDownloader, KaggleIngestionService

ingestion_settings = load_ingestion_settings(PROJECT_ROOT)
ingestion_result = KaggleIngestionService(
    settings=ingestion_settings,
    downloader=KaggleApiDownloader(),
).run()

display({
    'dataset_path': str(ingestion_result.dataset_path),
    'manifest_path': str(ingestion_result.manifest_path),
    'rows': ingestion_result.row_count,
    'sha256': ingestion_result.sha256,
    'snapshot_reutilizado': ingestion_result.reused_existing_snapshot,
})

## 2. Conectar ao MinIO e sincronizar o snapshot

Os objetos usam o mesmo SHA-256 do snapshot local. Um objeto existente só é reutilizado quando tamanho e metadados de integridade coincidem; divergências bloqueiam a execução em vez de sobrescrever dados.

In [ ]:
from obesity_risk_pipeline.config import load_minio_settings
from obesity_risk_pipeline.storage import MinioDatasetStore

minio_settings = load_minio_settings()
dataset_store = MinioDatasetStore(minio_settings)
remote_snapshot = dataset_store.ensure_snapshot(
    dataset_path=ingestion_result.dataset_path,
    manifest_path=ingestion_result.manifest_path,
    expected_sha256=ingestion_result.sha256,
)

display({
    'endpoint': minio_settings.endpoint,
    'bucket': remote_snapshot.bucket,
    'dataset_object': remote_snapshot.dataset_object,
    'manifest_object': remote_snapshot.manifest_object,
    'objetos_reutilizados': remote_snapshot.reused_existing_objects,
})

## 3. Ler o dataset diretamente do MinIO

A leitura verifica novamente o SHA-256 antes de entregar os bytes ao Pandas. `raw_df` preserva os nomes originais; `df` contém apenas a correção canônica do nome e do valor do alvo.

In [ ]:
dataset_bytes = dataset_store.read_verified_dataset(
    remote_snapshot.dataset_object,
    expected_sha256=remote_snapshot.sha256,
)
raw_df = pd.read_csv(BytesIO(dataset_bytes))
df = raw_df.rename(columns={'0be1dad': 'NObeyesdad'}).copy()
df['NObeyesdad'] = df['NObeyesdad'].replace({'0rmal_Weight': 'Normal_Weight'})

assert len(raw_df) == ingestion_result.row_count
assert raw_df.shape[1] == 18
display(df.head())

## 4. Estrutura, qualidade e cardinalidade

In [ ]:
quality_summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'cardinality': df.nunique(dropna=False),
}).sort_values(['missing', 'cardinality'], ascending=[False, True])

print(f'Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}'.replace(',', '.'))
print(f'Duplicatas sem id: {df.drop(columns=["id"]).duplicated().sum()}')
display(quality_summary)

In [ ]:
numeric_columns = [
    'Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE'
]
display(df[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T)

## 5. Distribuição do alvo

In [ ]:
target_order = [
    'Insufficient_Weight',
    'Normal_Weight',
    'Overweight_Level_I',
    'Overweight_Level_II',
    'Obesity_Type_I',
    'Obesity_Type_II',
    'Obesity_Type_III',
]
target_distribution = (
    df['NObeyesdad']
    .value_counts()
    .reindex(target_order)
    .rename_axis('classe')
    .reset_index(name='registros')
)
target_distribution['percentual'] = (
    target_distribution['registros'] / len(df) * 100
).round(2)
display(target_distribution)

plt.figure(figsize=(10, 5))
sns.barplot(data=target_distribution, x='registros', y='classe', color='#4C72B0')
plt.title('Distribuição das classes do alvo')
plt.xlabel('Registros')
plt.ylabel('Classe')
plt.tight_layout()
plt.show()

## 6. Relação antropométrica e possível circularidade

O IMC abaixo é derivado somente para auditoria. Como o alvo está fortemente relacionado a altura e peso, compare futuramente modelos com e sem essas variáveis para evitar conclusões artificialmente otimistas.

In [ ]:
analysis_df = df.assign(BMI=df['Weight'] / df['Height'].pow(2))
bmi_by_target = (
    analysis_df.groupby('NObeyesdad', observed=True)['BMI']
    .agg(['count', 'mean', 'median', 'min', 'max'])
    .reindex(target_order)
    .round(2)
)
display(bmi_by_target)

plt.figure(figsize=(11, 5))
sns.boxplot(data=analysis_df, x='NObeyesdad', y='BMI', order=target_order)
plt.xticks(rotation=35, ha='right')
plt.title('IMC derivado por classe')
plt.xlabel('Classe')
plt.ylabel('IMC derivado')
plt.tight_layout()
plt.show()

## 7. Auditoria inicial por gênero

In [ ]:
gender_by_target = (
    pd.crosstab(
        df['NObeyesdad'],
        df['Gender'],
        normalize='index',
    )
    .mul(100)
    .reindex(target_order)
    .round(2)
)
display(gender_by_target)

gender_by_target.plot(kind='bar', stacked=True, figsize=(11, 5), colormap='Set2')
plt.title('Composição percentual de gênero por classe')
plt.xlabel('Classe')
plt.ylabel('Percentual dentro da classe')
plt.xticks(rotation=35, ha='right')
plt.legend(title='Gender', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Próximas análises sugeridas

- Investigar categorias raras em `SMOKE`, `SCC` e `MTRANS`.
- Comparar distribuições numéricas e ordinais entre as sete classes.
- Procurar registros quase duplicados antes do split.
- Auditar a associação quase determinística entre gênero e algumas classes.
- Definir o objetivo real: reproduzir o estado de peso atual ou estimar suscetibilidade sem peso/IMC.
- Manter toda transformação reutilizável em módulos Python com `fit` restrito ao treino; o notebook deve permanecer exploratório.